# Live Demo: Function Calling with the Conversations API

A complete, end-to-end **function-calling loop** built on `client.conversations.create()`. We define a few example tools, let the model decide which to call, execute them locally, feed the results back into the **same conversation**, and get the final answer.

**The round-trip:**
1. Create a durable conversation.
2. Send a user turn + tool definitions.
3. Model emits one or more `function_call` items.
4. We execute each tool locally and append `function_call_output` items.
5. Send those outputs back (same conversation) -> model produces the final answer.

> **Model:** `gpt-5.5` (tool orchestration is reasoning-heavy). `reasoning_effort` is pinned to `medium`.
>
> **TODO — `gpt-5.5-instant` [unverified]:** the API name for the instant variant was unconfirmed at authoring time, so this notebook uses `gpt-5.5`.

In [ ]:
from openai import OpenAI
import json

client = OpenAI()

## 1. Define example tools

In [ ]:
# Three example tools the model can call.
tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Get the current weather for a city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "City name, e.g. 'Lisbon'"},
                "unit": {"type": "string", "enum": ["c", "f"], "description": "Temperature unit"},
            },
            "required": ["city"],
        },
    },
    {
        "type": "function",
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string", "description": "ISO code, e.g. 'USD'"},
                "to_currency": {"type": "string", "description": "ISO code, e.g. 'EUR'"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
    {
        "type": "function",
        "name": "get_time",
        "description": "Get the current local time in a city.",
        "parameters": {
            "type": "object",
            "properties": {"city": {"type": "string"}},
            "required": ["city"],
        },
    },
]

## 2. Implement the tools (local execution)

In [ ]:
# Real implementations would call external APIs; here we mock them deterministically.
def get_weather(city, unit="c"):
    temp_c = {"lisbon": 22, "london": 14, "tokyo": 18}.get(city.lower(), 20)
    temp = temp_c if unit == "c" else round(temp_c * 9 / 5 + 32)
    return {"city": city, "temp": temp, "unit": unit, "conditions": "partly cloudy"}

def convert_currency(amount, from_currency, to_currency):
    rates = {("USD", "EUR"): 0.92, ("EUR", "USD"): 1.09, ("USD", "JPY"): 156.0}
    rate = rates.get((from_currency.upper(), to_currency.upper()), 1.0)
    return {"amount": amount, "from": from_currency, "to": to_currency,
            "rate": rate, "converted": round(amount * rate, 2)}

def get_time(city):
    return {"city": city, "local_time": "14:35", "tz": "local-mock"}

# Dispatch table: tool name -> callable
TOOL_IMPL = {
    "get_weather": get_weather,
    "convert_currency": convert_currency,
    "get_time": get_time,
}

def run_tool(name, arguments_json):
    args = json.loads(arguments_json)
    result = TOOL_IMPL[name](**args)
    return json.dumps(result)

## 3. The function-calling loop (Conversations API)

In [ ]:
# Create a durable conversation that carries tool calls + outputs across turns.
conversation = client.conversations.create()

user_message = (
    "What's the weather in Lisbon, and how much is 100 USD in EUR? "
    "Also what's the local time there?"
)

def run_function_calling_turn(user_text, max_rounds=5):
    """Drive the model<->tools loop until the model stops asking for tools."""
    # First model turn with the user's request.
    response = client.responses.create(
        model="gpt-5.5",
        conversation=conversation.id,
        tools=tools,
        input=[{"role": "user", "content": user_text}],
        reasoning={"effort": "medium"},
    )

    for round_i in range(max_rounds):
        # Collect any function calls the model emitted this round.
        calls = [item for item in response.output if item.type == "function_call"]
        if not calls:
            return response  # model is done -> final answer is ready

        # Execute each requested tool and build the outputs to send back.
        tool_outputs = []
        for call in calls:
            print(f"  -> model called {call.name}({call.arguments})")
            output = run_tool(call.name, call.arguments)
            tool_outputs.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": output,
            })

        # Send tool outputs back on the SAME conversation; model continues reasoning.
        response = client.responses.create(
            model="gpt-5.5",
            conversation=conversation.id,
            tools=tools,
            input=tool_outputs,
            reasoning={"effort": "medium"},
        )

    return response  # safety stop after max_rounds

final = run_function_calling_turn(user_message)
print("\n=== FINAL ANSWER ===")
print(final.output_text)

## 4. What just happened

- We created one **durable conversation** and never rebuilt history by hand.
- The model emitted `function_call` items; we executed them locally and replied with `function_call_output` items keyed by `call_id`.
- The loop repeats until the model stops requesting tools — handling **parallel** and **multi-round** tool use automatically.
- Because everything is attached to the conversation id, the tool calls and outputs persist and stay in context for any follow-up turn.

Try a follow-up on the same conversation to see the persisted context in action:

In [ ]:
# Follow-up turn reuses the SAME conversation -- prior tool results are still in context.
followup = client.responses.create(
    model="gpt-5.5",
    conversation=conversation.id,
    tools=tools,
    input=[{"role": "user", "content": "Convert that same USD amount to JPY instead."}],
    reasoning={"effort": "medium"},
)
# It may call convert_currency again; drive one more loop if needed.
calls = [i for i in followup.output if i.type == "function_call"]
if calls:
    outs = [{"type": "function_call_output", "call_id": c.call_id,
             "output": run_tool(c.name, c.arguments)} for c in calls]
    followup = client.responses.create(
        model="gpt-5.5", conversation=conversation.id, tools=tools,
        input=outs, reasoning={"effort": "medium"},
    )
print(followup.output_text)